# Generalized spectral clustering

Clusters all 445 feature titles together, without filtering by product type. 

Our aim is to find attributes form groups that are strongly linked inside and only weakly linked to the rest.

We start from the shared similarity matrix: `3_similarity.ipynb` now saves it to `data/similarity.npz`, and this notebook (like the agglomerative and DBSCAN ones) just loads it.

**Method** (normalized spectral clustering):
1. We build a graph from the similarity matrix: attributes are nodes, similarities are edge weights (each node keeps its `K_NEIGHBORS` most similar neighbours).
2. We compute the normalized Laplacian and its eigenvectors with the smallest eigenvalues.
3. Finally, we run k-means on those eigenvectors

In [2]:
import json
import time
from collections import Counter

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import HTML, display
from sklearn.cluster import KMeans
from scipy.sparse.csgraph import connected_components
from sklearn.metrics import davies_bouldin_score, silhouette_score

#links kept per attribute
K_NEIGHBORS = 15

#how many coordinates to compute at most
N_EIGEN = 60
K_RANGE = range(5, 61, 5)

#to make random parts repeatable
SEED = 0

#loading the similarity matrix (showing how alike each pair of attributes is)
with np.load("data/similarity.npz") as data:
    S = data["similarity"]
    #list of the attributes in the same order as matched in the matrix
    attributes = data["titles"].tolist()


#preliminary checks on the similarity matrix
n = len(attributes)
assert S.shape == (n, n), f"matrix {S.shape} does not match {n} attributes"
assert len(set(attributes)) == n, "duplicate attribute names"
assert S.min() >= 0 and S.max() <= 1 + 1e-9, "values are not in the interval [0,1]"
assert np.allclose(S, S.T), "similarity must be symmetric"

print(f"Loaded {n} attributes and matrix {S.shape}")

Loaded 445 attributes and matrix (445, 445)


## Build the similarity graph

The diagonal (originally set as 1, since all attributes are obviously strongly similar (identical!) to themselves) is set to 0 (we want no self-loops to W and that a node's own entry doesn't take one of its k neighbour slots). 

With `K_NEIGHBORS`, each attribute keeps only its most similar neighbours; the result is symmetrised so that an edge is kept if either endpoint selects it. 

This makes the graph sparse, removes weak noisy links, and is what makes spectral clustering scale to larger catalogs.

In [3]:
def build_affinity(S, k_neighbors=K_NEIGHBORS):
    W = S.copy()
    np.fill_diagonal(W, 0)

    #sorts each row from smallest to largest, take k-th value from the end of row, turn values into a column
    threshold = np.sort(W, axis=1)[:, -k_neighbors][:, None]

    #we check each pair (x,y): if x is in y´s top K OR viceversa, the edge is kept
    #otherwise it is set to 0 
    W = np.where((W >= threshold) | (W >= threshold.T), W, 0) 
    return W


W = build_affinity(S, K_NEIGHBORS)

#check W is symmetric
assert np.allclose(W, W.T)

#weighted degree of d_i: sum of the similarities on node i's edges
deg = W.sum(axis=1)
#how many neighbors each node has 
nbrs = (W > 0).sum(axis=1)
#number of connected components
n_comp, _ = connected_components(W > 0, directed=False)

print(f"Edges kept: {np.count_nonzero(np.triu(W))} of {n*(n-1)//2} ({np.count_nonzero(np.triu(W))/(n*(n-1)//2):.1%})")

print(f"Neighbours per node: min {nbrs.min()}, median {int(np.median(nbrs))}, max {nbrs.max()}")

print(f"Weighted degree: min {deg.min():.2f}, median {np.median(deg):.2f}, max {deg.max():.2f}")

print(f"Isolated nodes: {(deg == 0).sum()}, connected components: {n_comp}")

Edges kept: 4618 of 98790 (4.7%)
Neighbours per node: min 15, median 18, max 59
Weighted degree: min 1.01, median 5.60, max 18.85
Isolated nodes: 0, connected components: 1


## Normalized Laplacian and eigengap

`np.linalg.eigh` is used because $L_{sym}$ is symmetric; it returns eigenvalues in ascending order. Small eigenvalues correspond to well-separated groups; the largest jump (eigengap) among them suggests the number of clusters.

In [ ]:
def normalized_laplacian(W):
    # matrix whose eigenvectors give the best ways to split the graph into groups
    #L_sym = I - D^(-1/2) W D^(-1/2): each link w_ij is rescaled to w_ij / sqrt(d_i * d_j)
    
    #without it, very connected attributes would dominate and pull everything towards them
    #after rescaling, a link counts more when it’s important to both sides
    
    deg = W.sum(axis=1)   

    #computing 1/sqrt(d) for each attribute of deg                       
    d_inv_sqrt = 1 / np.sqrt(np.maximum(deg, 1e-12)) # 1e-12 avoids division by zero
    #eye build the idientity matrix, outer builds a multiplication table having d_inv_sqrt as both a row and a column
    #w is multiplied element wise
    return np.eye(len(W)) - W * np.outer(d_inv_sqrt, d_inv_sqrt)


def spectral_embedding(W, n_eigen):
    """Return the n_eigen smallest eigenvalues of L_sym and their eigenvectors."""
    eigenvalues, eigenvectors = np.linalg.eigh(normalized_laplacian(W))
    return eigenvalues[:n_eigen], eigenvectors[:, :n_eigen]

K_MIN, K_MAX = 35, 35

eigenvalues, eigenvectors = spectral_embedding(W, n_eigen=K_MAX + 1)

gaps = np.diff(eigenvalues)

# number of small eigenvalues before the biggest jump = number of clusters
k_eigengap = int(np.argmax(gaps[K_MIN - 1:K_MAX]) + K_MIN)

print(f"Eigengap between k={K_MIN} and {K_MAX} suggests k = {k_eigengap} "
      f"(gap = {gaps[k_eigengap - 1]:.4f})")

Eigengap between k=20 and 80 suggests k = 25 (gap = 0.0286)


## 4. Spectral clustering

Rows of the first *k* eigenvectors are normalized to unit length (Ng–Jordan–Weiss step), then clustered with k-means.

In [5]:
def spectral_cluster(eigenvectors, k, seed=SEED):
    U = eigenvectors[:, :k]
    U = U / np.maximum(np.linalg.norm(U, axis=1, keepdims=True), 1e-12)
    labels = KMeans(n_clusters=k, n_init=10, random_state=seed).fit_predict(U)
    return labels, U


k = k_eigengap
labels, U = spectral_cluster(eigenvectors, k)
sizes = Counter(labels)
print(f"{k} clusters; sizes (largest first): {[size for _, size in sizes.most_common()]}")

from collections import Counter

#to be deleted after rename
feature_names = attributes

cluster_sizes = Counter(labels.tolist())
cluster_labels = sorted(cluster_sizes)
singleton_clusters = [cluster_label for cluster_label in cluster_labels if cluster_sizes[cluster_label] == 1]

print(
    f"Clusters: {len(cluster_labels)} | "
    f"singleton clusters: {len(singleton_clusters)} / {len(feature_names)} feature names "
    f"({100 * len(singleton_clusters) / len(feature_names):.1f}%)"
)
print(f"Cluster sizes (largest first): {[size for _, size in cluster_sizes.most_common()]}")

feature_names_by_cluster = {
    cluster_label: [
        feature_names[index]
        for index, assigned_cluster_label in enumerate(labels)
        if assigned_cluster_label == cluster_label
    ]
    for cluster_label in cluster_labels
}

for cluster_label, cluster_feature_names in sorted(
    feature_names_by_cluster.items(), key=lambda item: (-len(item[1]), item[0])
):
    if len(cluster_feature_names) > 1:
        print(f"\nCluster {cluster_label} ({len(cluster_feature_names)} feature names)")
        for feature_name in cluster_feature_names:
            print(f"  - {feature_name}")

25 clusters; sizes (largest first): [31, 30, 26, 24, 22, 21, 21, 20, 19, 18, 18, 18, 17, 17, 17, 17, 14, 14, 14, 13, 12, 11, 11, 11, 9]
Clusters: 25 | singleton clusters: 0 / 445 feature names (0.0%)
Cluster sizes (largest first): [31, 30, 26, 24, 22, 21, 21, 20, 19, 18, 18, 18, 17, 17, 17, 17, 14, 14, 14, 13, 12, 11, 11, 11, 9]

Cluster 1 (31 feature names)
  - High stretch
  - High stretch and anatomically optimised fit
  - Selective compression
  - Stretch fabric
  - Stretch arch support
  - Ultra-Soft Thermal Fabric
  - Stretch arch support for superior fit
  - Premium high-stretch material back
  - Stretch Arch Support
  - Ergonomic construction
  - High stretch and secure fit
  - Performance fabrics
  - Super stretch
  - Stretchable fleece inserts
  - Comfort and performance
  - Soft and stretchy
  - Active Fit
  - Compressive performance materials
  - Thermal stretch fabric
  - Raglan sleeves and articulated arms
  - Extended back length
  - Woven 4-way stretch fabric
  - Highly

## 5. Evaluation

## 5. Evaluation

Do the clusters actually group attributes that are similar to each other? To answer it fairly, we evaluate every result on the original similarities `S` (or the distances `1 - S`). This is the common ground all methods start from: the agglomerative, DBSCAN and HDBSCAN notebooks receive exactly the same matrix and have no graph or embedding of their own. 

- 5.1 Internal quality (silhouette, Davies-Bouldin, modularity)
- 5.2 Choosing *k*
- 5.3 Coherence (to be completed)
- 5.4 Outliers, density and radius
- 5.5 Runtime
- 5.6 Stability and correctness

### 5.1 Internal quality

Here we check how good the clusters produced by spectral clustering are. We take the labels from section 4 and measure them against the shared similarity matrix `S`, which stays fixed.

The **silhouette** looks at single attributes. For each one, it asks whether the attribute is closer to the other members of its own cluster than to the nearest other cluster, with score going from −1 to 1.

The **Davies-Bouldin index** looks at pairs of clusters. For each cluster, it finds the neighbouring cluster it overlaps with the most, and compares how spread out the two clusters are with how far apart they are.

**Modularity** looks at the links. It compares the similarity inside the clusters with the similarity we would expect if links were spread at random.

In [6]:
from sklearn.metrics import silhouette_samples, adjusted_rand_score   # used in 5.4 and 5.6
from sklearn.cluster import SpectralClustering                        # used in 5.6

# Shared inputs
D = 1 - S
np.fill_diagonal(D, 0)
G = S.copy()       
np.fill_diagonal(G, 0)


def clustroid(labels):
    #to find each cluster's most central member
    result = {}
    for label in np.unique(labels):
        members = np.where(labels == label)[0]
        result[label] = members[D[np.ix_(members, members)].sum(axis=1).argmin()]
    return result


def silhouette(labels):
    return float(silhouette_score(D, labels, metric="precomputed"))


def davies_bouldin(labels):
    #find each cluster center
    centre = clustroid(labels)
    clusters = list(centre)
    #compute avg distance of members of a cluster to its clustroid
    spread = np.array([D[labels == c, centre[c]].mean() for c in clusters])
    centres = [centre[c] for c in clusters]

    separation = D[np.ix_(centres, centres)] + np.diag([np.inf] * len(clusters))
    #for every pair of clusters we compute (spread_i + spread_j) / distance between their centres
    ratios = (spread[:, None] + spread[None, :]) / separation
    #for each cluster we keep its worst ratio (its most overlapping neighbour)
    worst = ratios.max(axis=1)
    #computing average over all clusters
    return float(worst.mean())

def modularity(labels):
    #degree of each attribute
    degree = G.sum(axis=1)
    two_m = degree.sum()
    #true-false table to find pairs in the same cluster
    same = labels[:, None] == labels[None, :]
    return float(((G - np.outer(degree, degree) / two_m) * same).sum() / two_m)


def evaluate(labels):
    return {"silhouette": silhouette(labels),
            "davies_bouldin": davies_bouldin(labels),
            "modularity": modularity(labels)}


scores = evaluate(labels)
print(f"k = {k}")
for name, value in scores.items():
    print(f"  {name:15} {value:.3f}")

k = 25
  silhouette      0.122
  davies_bouldin  1.653
  modularity      0.124


### 5.2 Choosing *k*

We sweep *k* from 10 to 100, keeping the graph and the eigenvectors fixed, and plot the three measures plus the size of the largest and smallest cluster. The dashed line marks the eigengap choice. A good *k* has a high silhouette and modularity, a low Davies-Bouldin, and no oversized catch-all cluster.

In [7]:
K_SWEEP = range(10, 101, 5)
_, eigenvectors_sweep = spectral_embedding(W, n_eigen=max(K_SWEEP))

# Cluster and evaluate for every k
sweep = []
for k_ in K_SWEEP:
    labels_k, _ = spectral_cluster(eigenvectors_sweep, k_)
    sizes_k = Counter(labels_k).values()
    sweep.append({"k": k_, **evaluate(labels_k), "largest": max(sizes_k), "smallest": min(sizes_k)})

# Table
print(f"{'k':>4} {'silhouette':>11} {'DB':>6} {'modularity':>11} {'largest':>8} {'smallest':>9}")
for r in sweep:
    print(f"{r['k']:>4} {r['silhouette']:>11.3f} {r['davies_bouldin']:>6.2f} "
          f"{r['modularity']:>11.3f} {r['largest']:>8} {r['smallest']:>9}")

# Plot
ks = [r["k"] for r in sweep]
panels = {"Silhouette ↑": ["silhouette"], "Davies-Bouldin ↓": ["davies_bouldin"],
          "Modularity ↑": ["modularity"], "Cluster size": ["largest", "smallest"]}

figure = make_subplots(rows=2, cols=2, subplot_titles=list(panels))
for i, metrics in enumerate(panels.values()):
    for metric in metrics:
        figure.add_trace(go.Scatter(x=ks, y=[r[metric] for r in sweep], mode="lines+markers", name=metric),
                         row=i // 2 + 1, col=i % 2 + 1)
figure.add_vline(x=k_eigengap, line_dash="dash", line_color="gray", row="all", col="all")
figure.update_xaxes(title_text="k")
figure.update_layout(template="simple_white", width=1000, height=650, showlegend=False,
                     title="Spectral clustering: quality vs number of clusters")
display(HTML(figure.to_html(full_html=False, include_plotlyjs="cdn")))

   k  silhouette     DB  modularity  largest  smallest
  10       0.088   1.65       0.191      108        14
  15       0.101   1.69       0.164       73        10
  20       0.111   1.70       0.138       34        10
  25       0.122   1.65       0.124       31         9
  30       0.134   1.66       0.117       24         7
  35       0.137   1.59       0.104       24         7
  40       0.132   1.60       0.096       23         6
  45       0.146   1.57       0.090       16         4
  50       0.148   1.53       0.085       15         3
  55       0.135   1.54       0.077       15         3
  60       0.145   1.50       0.073       13         3
  65       0.136   1.58       0.070       13         3
  70       0.145   1.52       0.066       14         2
  75       0.145   1.44       0.063       14         2
  80       0.145   1.40       0.058       12         2
  85       0.139   1.43       0.055       11         1
  90       0.142   1.38       0.053       11         2
  95      

### 5.3 Coherence: does each cluster describe one property?

_TO BE COMPLETED_

Hand-labelled pairs or manual inspection?

### 5.4 Outliers, density and radius

Here we look inside each cluster. Its **density** is the average similarity between its members: the higher it is, the tighter the group. Its **radius** is the distance from the clustroid to the member furthest away, which shows how far the cluster stretches; we also report the average distance to the clustroid. Finally, we count the **outliers**: members with a negative silhouette, which are on average closer to another cluster than to their own.

In [8]:
silhouette_per_attribute = silhouette_samples(D, labels, metric="precomputed")

# Index of each cluster's clustroid
centre = clustroid(labels)

#we show one row of statistics per cluster
rows = []
for label, size in sizes.most_common():
    members = np.where(labels == label)[0]
    within = S[np.ix_(members, members)]

    #density formula
    density = (within.sum() - size) / (size * (size - 1)) if size > 1 else 1.0   # mean off-diagonal similarity
    to_clustroid = D[members, centre[label]]
    rows.append({
        "cluster": label, "size": size, "density": density,
        "mean_dist": to_clustroid.mean(), "radius": to_clustroid.max(),
        "outliers": int((silhouette_per_attribute[members] < 0).sum()),
        "clustroid": attributes[centre[label]],
    })

print(f"{'cluster':>7} {'size':>5} {'density':>8} {'mean dist':>10} {'radius':>7} {'outliers':>9}  clustroid")
for row in rows:
    print(f"{row['cluster']:>7} {row['size']:>5} {row['density']:>8.3f} {row['mean_dist']:>10.3f} "
          f"{row['radius']:>7.3f} {row['outliers']:>9}  {row['clustroid']}")

print("\nQuartiles across clusters (25% / 50% / 75%):")
for metric in ["size", "density", "mean_dist", "radius"]:
    q = np.percentile([row[metric] for row in rows], [25, 50, 75])
    print(f"  {metric:10} {q[0]:.3f} / {q[1]:.3f} / {q[2]:.3f}")

outliers = np.where(silhouette_per_attribute < 0)[0]
singletons = sum(1 for size in sizes.values() if size == 1)
print(f"\nOutliers (negative silhouette): {len(outliers)} of {n} ({len(outliers) / n:.1%}); singletons: {singletons}")

print("\nWorst 10 attributes (lowest silhouette):")
for i in np.argsort(silhouette_per_attribute)[:10]:
    print(f"  {silhouette_per_attribute[i]:6.3f}  {attributes[i]!r}  in cluster '{attributes[centre[labels[i]]]}'")

cluster  size  density  mean dist  radius  outliers  clustroid
      1    31    0.228      0.662   0.910         1  High stretch
     15    30    0.128      0.750   0.994        12  Close-fitting cut
      2    26    0.317      0.532   0.876         1  Highly breathable
     12    24    0.325      0.540   0.850         0  Insulating
      3    22    0.204      0.674   1.000         8  Merino wool fabric blend
     16    21    0.228      0.637   0.833         2  Road-specific fit
      6    21    0.257      0.586   0.828         1  Moisture wicking & quick-drying
      0    20    0.192      0.684   0.925         2  Hook-and-loop closure
      7    19    0.312      0.575   0.876         0  Silicone palm grip
     17    18    0.224      0.606   0.826         8  Coated 3 mm neoprene
     14    18    0.256      0.614   0.855         1  Durable and reliable
     20    18    0.186      0.698   0.970         2  Versatile multi-sport use
     10    17    0.332      0.483   0.743         3  Wind

### 5.5 Runtime

We timed the three stages of the pipeline (graph construction, eigendecomposition, k-means) on random subsets of increasing size, with the same `K_NEIGHBORS` and *k*. Each size is run 3 times and  we kept the median value

**Complexity:** _TO BE COMPLETED_

In [9]:
SUBSET_SIZES = [50, 100, 200, 300, n]  
REPEATS = 3                            
rng = np.random.default_rng(SEED)


def time_once(size):
    subset = rng.choice(n, size, replace=False)
    t0 = time.perf_counter()
    W_sub = build_affinity(S[np.ix_(subset, subset)], K_NEIGHBORS)  
    t1 = time.perf_counter()
    _, vectors = spectral_embedding(W_sub, n_eigen=k)               
    t2 = time.perf_counter()
    spectral_cluster(vectors, k)                                      
    t3 = time.perf_counter()
    return [t1 - t0, t2 - t1, t3 - t2]


stages = ["graph", "eigendecomposition", "k-means"]
# for each size we compute median over the repeats
timings = np.array([np.median([time_once(size) for _ in range(REPEATS)], axis=0) for size in SUBSET_SIZES])
total = timings.sum(axis=1)

# Table
print(f"{'n':>5} " + "".join(f"{s:>20}" for s in stages) + f"{'total':>10}")
for size, row, t in zip(SUBSET_SIZES, timings, total):
    print(f"{size:>5} " + "".join(f"{v:>19.4f}s" for v in row) + f"{t:>9.4f}s")

# Plot
figure = go.Figure()
for stage, values in zip(stages + ["total"], [*timings.T, total]):
    figure.add_trace(go.Scatter(x=SUBSET_SIZES, y=values, mode="lines+markers", name=stage,
                                line=dict(dash="dash" if stage == "total" else "solid")))
figure.update_layout(title="Spectral clustering runtime by stage", xaxis_title="number of attributes",
                     yaxis_title="seconds (median of 3)", template="simple_white", width=800, height=450)
display(HTML(figure.to_html(full_html=False, include_plotlyjs="cdn")))

    n                graph  eigendecomposition             k-means     total
   50              0.0002s             0.0021s             0.0998s   0.1021s
  100              0.0009s             0.0090s             0.1065s   0.1164s
  200              0.0030s             0.0140s             0.1283s   0.1454s
  300              0.0051s             0.0592s             0.1445s   0.2088s
  445              0.0120s             0.1353s             0.1550s   0.3022s


## 6. Cluster visualization

t-SNE projects the same precomputed distances (`1 - similarity`) to two dimensions for display only; it does not change the clustering labels or serve as an evaluation metric. Nearby points suggest similarity, but the projection can distort some distances. The largest groups get distinct colours, the others are grey, and singletons (if any) are hollow markers. Hover over a point to see its attribute and group.

In [10]:
from sklearn.manifold import TSNE

HIGHLIGHT_GROUPS = 8
HIGHLIGHT_PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
GRAY, LIGHT_GRAY, INK, MUTED = "#8c8c86", "#c9c8c0", "#1f2933", "#6b7785"

distance_matrix = 1 - S
multi = [label for label, size in sizes.items() if size > 1]

coordinates = TSNE(
    n_components=2,
    metric="precomputed",
    init="random",
    perplexity=30,
    random_state=42,
).fit_transform(distance_matrix)


def group_name(label):
    """Use the most centrally similar attribute as the group's display name."""
    indices = np.where(labels == label)[0]
    within_group_similarity = S[np.ix_(indices, indices)]
    clustroid_index = indices[within_group_similarity.sum(axis=1).argmax()]
    return attributes[clustroid_index]


ranked_groups = sorted(multi, key=lambda label: -sizes[label])
highlighted_groups = ranked_groups[:HIGHLIGHT_GROUPS]
group_names = {label: group_name(label) for label in multi}


def make_trace(indices, name, color, hollow=False, size=9):
    return go.Scatter(
        x=coordinates[indices, 0],
        y=coordinates[indices, 1],
        mode="markers",
        name=name,
        marker=dict(
            size=size,
            color="white" if hollow else color,
            line=dict(width=1.5 if hollow else 1, color=color if hollow else "white"),
        ),
        text=[attributes[index] for index in indices],
        customdata=[group_names.get(labels[index], "Singleton") for index in indices],
        hovertemplate="<b>%{text}</b><br>group: %{customdata}<extra></extra>",
    )


other_group_indices = np.array(
    [index for index, label in enumerate(labels) if sizes[label] > 1 and label not in highlighted_groups],
    dtype=int,
)
singleton_indices = np.array([index for index, label in enumerate(labels) if sizes[label] == 1], dtype=int)

figure = go.Figure()
figure.add_trace(
    make_trace(singleton_indices, f"Singletons ({len(singleton_indices)})", LIGHT_GRAY, hollow=True, size=7)
)
figure.add_trace(
    make_trace(other_group_indices, f"Other groups ({len(ranked_groups) - len(highlighted_groups)})", GRAY, size=7)
)
for color, label in zip(HIGHLIGHT_PALETTE, highlighted_groups):
    indices = np.where(labels == label)[0]
    figure.add_trace(make_trace(indices, f"{group_names[label]} ({sizes[label]})", color, size=10))

figure.update_layout(
    title=dict(text="Attribute clusters: spectral clustering (t-SNE)", x=0, font=dict(color=INK)),
    template="simple_white",
    width=1000,
    height=720,
    xaxis=dict(visible=False),
    yaxis=dict(visible=False),
    legend=dict(title="Largest groups", font=dict(color=INK), itemsizing="constant"),
    hoverlabel=dict(bgcolor="white", font_color=INK),
)
display(HTML(figure.to_html(full_html=False, include_plotlyjs="cdn")))